
# Notebook 09 · Resolución de identidad y customer 360 accionable

## Pregunta de negocio
¿Cómo unificamos cliente online, tienda y fidelización para construir una vista cliente 360 que sirva para segmentar, activar y medir mejor?

## Qué aprenderás
- Qué señales ayudan realmente a resolver identidad.
- Cómo evaluar una lógica de matching con un *ground truth*.
- Cómo traducir el resultado a un customer 360 activable.

## Outputs
- `nb09_identity_mapping.csv`
- `nb09_customer360.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

ids = tables["ids"].copy()
gt = tables["gt"].copy()
crm = tables["crm"].copy()
fact = tables["fact"].copy()
eventos_web = tables["eventos_web"].copy()

print("Identidades:", ids.shape)
print("CRM:", crm.shape)
print("Fact transacciones:", fact.shape)
print("Eventos web:", eventos_web.shape)

link_cols = ["email_hash", "telefono_hash", "tarjeta_fidelizacion_id", "cookie_id", "device_id", "app_user_id", "usuario_online_id"]
coverage = pd.DataFrame({
    "signal_type": link_cols,
    "non_null_rows": [ids[c].notna().sum() for c in link_cols],
    "distinct_keys": [ids[c].dropna().astype(str).nunique() for c in link_cols],
    "repeated_keys": [(ids[c].dropna().astype(str).value_counts() > 1).sum() for c in link_cols],
})
coverage.sort_values("repeated_keys", ascending=False)


Identidades: (411, 18)
CRM: (103, 24)
Fact transacciones: (4973, 61)
Eventos web: (9030, 23)


,signal_type,non_null_rows,distinct_keys,repeated_keys
0,email_hash,266,92,78
4,device_id,191,85,63
3,cookie_id,191,85,63
6,usuario_online_id,191,85,63
2,tarjeta_fidelizacion_id,183,83,59
1,telefono_hash,160,87,49
5,app_user_id,99,41,35


In [3]:

clusters = mpu.build_identity_clusters(ids)

mapping = (
    ids.merge(clusters, on="identity_signal_id", how="left")
      .merge(gt, on="identity_signal_id", how="left")
)

cluster_customer = (
    mapping.dropna(subset=["customer_id_raw"])
           .groupby(["pred_cluster_id", "customer_id_raw"])
           .size()
           .reset_index(name="n")
           .sort_values(["pred_cluster_id", "n"], ascending=[True, False])
           .drop_duplicates("pred_cluster_id")
           .rename(columns={"customer_id_raw": "pred_customer_id"})
)

mapping = mapping.merge(cluster_customer[["pred_cluster_id", "pred_customer_id"]], on="pred_cluster_id", how="left")
mapping["pred_customer_id"] = mapping["pred_customer_id"].fillna("SIN_ID_RAW")

cluster_purity = (
    mapping.dropna(subset=["customer_id_canonico"])
           .groupby("pred_cluster_id")["customer_id_canonico"]
           .agg(lambda s: s.value_counts(normalize=True).iloc[0])
           .rename("cluster_purity")
           .reset_index()
)

mapping = mapping.merge(cluster_purity, on="pred_cluster_id", how="left")

eval_accuracy = (
    mapping["pred_customer_id"].fillna("").str.upper().eq(
        mapping["customer_id_canonico"].fillna("").str.upper()
    )
).mean()

summary_identity = pd.DataFrame({
    "metric": [
        "signals",
        "predicted_clusters",
        "average_cluster_size",
        "median_cluster_size",
        "exact_customer_match_accuracy",
        "mean_cluster_purity"
    ],
    "value": [
        len(mapping),
        mapping["pred_cluster_id"].nunique(),
        round(mapping["cluster_size"].mean(), 2),
        round(mapping["cluster_size"].median(), 2),
        round(float(eval_accuracy), 4),
        round(float(mapping["cluster_purity"].mean()), 4)
    ]
})
summary_identity


,metric,value
0,signals,411.0000
1,predicted_clusters,143.0000
2,average_cluster_size,4.0700
3,median_cluster_size,4.0000
4,exact_customer_match_accuracy,0.9416
5,mean_cluster_purity,0.9927


In [4]:

sales = fact[fact["tipo_movimiento"] == "VENTA"].copy()

cust_value = (
    sales.groupby("customer_id")
         .agg(
             revenue_total_eur=("importe_abs_num", "sum"),
             margen_bruto_total_eur=("margen_bruto_num", "sum"),
             transacciones=("doc_id_std", "nunique"),
             categorias=("categoria", "nunique"),
             primera_compra=("fecha_movimiento_dt", "min"),
             ultima_compra=("fecha_movimiento_dt", "max"),
             revenue_online=("importe_abs_num", lambda s: s[sales.loc[s.index, "canal_movimiento"].eq("ONLINE")].sum()),
             revenue_pos=("importe_abs_num", lambda s: s[sales.loc[s.index, "canal_movimiento"].eq("POS")].sum()),
         )
         .reset_index()
)
cust_value["omnichannel_flag"] = ((cust_value["revenue_online"] > 0) & (cust_value["revenue_pos"] > 0)).astype(int)

web_agg = (
    eventos_web.groupby("customer_id")
              .agg(
                  sesiones=("session_id", "nunique"),
                  pageviews=("event_id", "count"),
                  add_to_cart=("tipo_evento", lambda s: (s == "add_to_cart").sum()),
                  searches=("tipo_evento", lambda s: (s == "search").sum()),
                  last_web_event=("fecha_hora_evento_dt", "max")
              )
              .reset_index()
)

customer360 = (
    mapping[["pred_cluster_id", "pred_customer_id"]]
      .drop_duplicates()
      .rename(columns={"pred_customer_id": "customer_id"})
      .merge(crm, on="customer_id", how="left")
      .merge(cust_value, on="customer_id", how="left")
      .merge(web_agg, on="customer_id", how="left")
)

customer360["days_since_last_purchase"] = (sales["fecha_movimiento_dt"].max() - customer360["ultima_compra"]).dt.days
customer360 = customer360.sort_values(["omnichannel_flag", "revenue_total_eur"], ascending=[False, False])

customer360.head(10)


,pred_cluster_id,customer_id,nombre,apellidos,email,telefono,fecha_alta_programa,tier_fidelizacion,puntos_acumulados,fecha_nacimiento,genero,ciudad,provincia,codigo_postal,pais,consentimiento_marketing,estado_cliente,fecha_baja,origen_alta,canal_preferido_declarado,ultima_actualizacion,fecha_alta_programa_dt,fecha_baja_dt,ultima_actualizacion_dt,fecha_nacimiento_dt,revenue_total_eur,margen_bruto_total_eur,transacciones,categorias,primera_compra,ultima_compra,revenue_online,revenue_pos,omnichannel_flag,sesiones,pageviews,add_to_cart,searches,last_web_event,days_since_last_purchase
65,IDS-000077,C0077,BRENDA,RuizPérez,brenda.ruizperez@example.com,631-690-609,2020-11-27,BRONCE,3130,1989-02-16,No binario,San Sebastián,Gipuzkoa,20003.0,España,S,Inactivo,2023-11-19,online,Online,24/04/25 16:00,NaT,2023-11-19,2025-04-24 16:00:00,NaT,38802.51,17122.66,29.0,6.0,2024-01-02 20:56:26,2025-11-07 12:14:00,20818.81,17983.70,1.0,24.0,102.0,10.0,15.0,2025-10-11 17:32:00,1727.0
66,IDS-000077,C0077,BRENDA,RuizPérez,brenda.ruizperez@example.com,631-690-609,32/01/2020,BRONCE,1.250,1989-02-16,No binario,San Sebastián,Gipuzkoa,20003.0,España,NaN,Activo,2023-11-19,online,Online,24/04/25 16:00,NaT,2023-11-19,2025-04-24 16:00:00,NaT,38802.51,17122.66,29.0,6.0,2024-01-02 20:56:26,2025-11-07 12:14:00,20818.81,17983.70,1.0,24.0,102.0,10.0,15.0,2025-10-11 17:32:00,1727.0
118,IDS-000381,C0077,BRENDA,RuizPérez,brenda.ruizperez@example.com,631-690-609,2020-11-27,BRONCE,3130,1989-02-16,No binario,San Sebastián,Gipuzkoa,20003.0,España,S,Inactivo,2023-11-19,online,Online,24/04/25 16:00,NaT,2023-11-19,2025-04-24 16:00:00,NaT,38802.51,17122.66,29.0,6.0,2024-01-02 20:56:26,2025-11-07 12:14:00,20818.81,17983.70,1.0,24.0,102.0,10.0,15.0,2025-10-11 17:32:00,1727.0
119,IDS-000381,C0077,BRENDA,RuizPérez,brenda.ruizperez@example.com,631-690-609,32/01/2020,BRONCE,1.250,1989-02-16,No binario,San Sebastián,Gipuzkoa,20003.0,España,NaN,Activo,2023-11-19,online,Online,24/04/25 16:00,NaT,2023-11-19,2025-04-24 16:00:00,NaT,38802.51,17122.66,29.0,6.0,2024-01-02 20:56:26,2025-11-07 12:14:00,20818.81,17983.70,1.0,24.0,102.0,10.0,15.0,2025-10-11 17:32:00,1727.0
79,IDS-000053,C0053,NURIA,AlonsoGonzález,nuria.alonsogonzalez@example.com,67O239146,2024-02-04,Bronce,3257,NaN,No binario,Vigo,Pontevedra,36201.0,NaN,sí,Inactivo,10 de Abril de 2025,importación,Online,2025-07-22 03:11:08,NaT,NaT,2025-07-22 03:11:08,NaT,38614.42,10511.94,10.0,4.0,2024-09-03 01:21:30,2025-09-06 07:54:00,4514.74,34099.68,1.0,20.0,75.0,4.0,11.0,2025-09-08 12:25:00,1789.0
58,IDS-000025,C0025,Yago,GilMartínez,NaN,+34 769 425 995,22 de Junio de 2023,Oro,4220,2004-02-15,No binario,Alicante,Alicante,3001.0,ES,S,Activo,NaN,online,Mixto,2025-06-12 13:09:15,NaT,NaT,2025-06-12 13:09:15,NaT,19748.90,7069.80,15.0,6.0,2025-01-08 01:53:00,2025-04-02 00:49:00,10362.84,9386.06,1.0,16.0,68.0,7.0,10.0,2025-06-10 21:59:00,1946.0
72,IDS-000288,C0025,Yago,GilMartínez,NaN,+34 769 425 995,22 de Junio de 2023,Oro,4220,2004-02-15,No binario,Alicante,Alicante,3001.0,ES,S,Activo,NaN,online,Mixto,2025-06-12 13:09:15,NaT,NaT,2025-06-12 13:09:15,NaT,19748.90,7069.80,15.0,6.0,2025-01-08 01:53:00,2025-04-02 00:49:00,10362.84,9386.06,1.0,16.0,68.0,7.0,10.0,2025-06-10 21:59:00,1946.0
19,IDS-000021,C0021,Samuel,PérezOrtiz,samuel.perezortiz@example.com,717884357,11 de Noviembre de 2019,Oro,4455,NaN,M,Bilbao,Bizkaia,NaN,ES,YES,Bloqueado,2023-02-22,online,Mixto,2025-10-02 05:30:33,NaT,2023-02-22,2025-10-02 05:30:33,NaT,18789.64,6405.66,14.0,6.0,2025-02-10 17:24:00,2025-12-03 21:44:01,10132.97,8656.67,1.0,16.0,70.0,6.0,9.0,2025-04-04 15:27:00,1701.0
120,IDS-000101,C0047,Mateo,FernándezOrtega,mateo.fernandezortega@example.com,+34 711 567 612,2025-09-06,Plata,2431,05/07/1971,NaN,Valladolid,Valladolid,47001.0,ES,1,Activo,NaN,NaN,Online,2025-05-15 07:09:06,NaT,NaT,2025-05-15 07:09:06,1971-07-05,18723.50,5382.51,11.0,6.0,2024-09-04 22:27:00,2025-10-02 02:25:52,5471.50,13252.00,1.0,13.0,47.0,2.0,5.0,2025-09-02 21:59:00,1763.0
121,IDS-000101,C0047,Mateo,Fer

In [5]:

duplicates_resolved = (
    mapping.groupby("pred_cluster_id")
           .agg(
               signals=("identity_signal_id", "nunique"),
               raw_customer_ids=("customer_id_raw", lambda s: s.dropna().astype(str).nunique()),
               example_raw_ids=("customer_id_raw", lambda s: ", ".join(sorted(s.dropna().astype(str).unique())[:5])),
               example_sources=("fuente_sistema", lambda s: ", ".join(sorted(s.dropna().astype(str).unique())[:5])),
               canonical_gt=("customer_id_canonico", lambda s: ", ".join(sorted(s.dropna().astype(str).unique())[:3])),
               cluster_purity=("cluster_purity", "max")
           )
           .reset_index()
           .query("signals > 1")
           .sort_values(["raw_customer_ids", "signals", "cluster_purity"], ascending=[False, False, False])
)
duplicates_resolved.head(15)


,pred_cluster_id,signals,raw_customer_ids,example_raw_ids,example_sources,canonical_gt,cluster_purity
66,IDS-000088,9,2,"C0060, C0088","APP, CRM, ECOM, POS","C0060, C0088",0.666667
42,IDS-000059,8,1,C0059,"CRM, ECOM, POS",C0059,1.000000
57,IDS-000077,7,1,C0077,"CRM, ECOM, POS",C0077,1.000000
1,IDS-000002,6,1,C0002,"APP, CRM, ECOM, POS",C0002,1.000000
44,IDS-000062,6,1,C0062,"CRM, ECOM, POS",C0062,1.000000
52,IDS-000071,6,1,C0071,"APP, CRM, ECOM",C0071,1.000000
61,IDS-000083,6,1,C0083,"APP, CRM, ECOM, POS",C0083,1.000000
68,IDS-000091,6,1,C0091,"CRM, ECOM, POS",C0091,1.000000
71,IDS-000094,6,1,C0094,"CRM, ECOM, POS",C0094,1.000000
81,IDS-000153,6,1,C0074,"CRM, ECOM, POS",C0074,1.000000



## Lectura de negocio

Un matching aceptable no termina en la tasa de acierto. El valor aparece cuando el resultado permite:

1. dejar de sobrecontar clientes y ventas anónimas,
2. consolidar actividad omnicanal por persona,
3. mejorar el targeting de campañas,
4. medir mejor frecuencia, CLV y churn.

En un entorno de marketing de precisión, este notebook es la base para todos los casos siguientes.


In [6]:

out_map = mapping.sort_values(["pred_cluster_id", "identity_signal_id"]).copy()
out_360 = customer360.copy()

path_map = mpu.export_csv(out_map, "nb09_identity_mapping.csv")
path_360 = mpu.export_csv(out_360, "nb09_customer360.csv")

print("Guardado:", path_map)
print("Guardado:", path_360)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb09_identity_mapping.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb09_customer360.csv
